# Semaine 2 — Jour 5 : Memory courte, longue et state

Ce notebook est la version étudiant du jour 5.

Il accompagne les fichiers Markdown de référence dans `book/week02/day05/`.

Objectif : comprendre et tester une architecture mémoire minimale pour un agent IA.

## 1. Modèle mental

Un agent mémoire-aware sépare trois responsabilités :

| Couche | Rôle |
|---|---|
| Short-term memory | Conserver les derniers échanges utiles |
| Conversation state | Suivre la tâche en cours |
| Long-term memory | Conserver des préférences ou faits stables par utilisateur |

La mémoire longue ne doit pas contenir tout l'historique.

## 2. Charger le lab

La cellule suivante cherche automatiquement le dossier du lab dans le dépôt.

In [ ]:
from pathlib import Path
import sys

def find_lab_path() -> Path:
    for candidate_root in [Path.cwd(), *Path.cwd().parents]:
        lab_path = candidate_root / "book" / "week02" / "day05" / "labs"
        if (lab_path / "memory_agent.py").exists():
            return lab_path
    raise FileNotFoundError("Impossible de trouver book/week02/day05/labs")

LAB_PATH = find_lab_path()
sys.path.insert(0, str(LAB_PATH))

from memory_agent import MemoryAwareSupportAgent, LongTermMemoryStore

LAB_PATH

## 3. Première interaction

On initialise un agent déterministe.

Aucun appel réseau n'est effectué.

In [ ]:
agent = MemoryAwareSupportAgent(short_term_max_messages=4)

print(agent.receive("user_1", "Tu peux m'appeler Nadia."))
print(agent.receive("user_1", "Je préfère les exemples en Python et les réponses courtes."))
print(agent.receive("user_1", "J'ai un problème avec Billing API."))

## 4. Observer le contexte construit

Le contexte contient :

- le profil utilisateur ;
- l'état courant ;
- les messages récents ;
- un résumé court.

In [ ]:
import json

context = agent.build_context("user_1")
print(json.dumps(context, ensure_ascii=False, indent=2))

## 5. Isolation utilisateur

Une mémoire professionnelle ne doit jamais mélanger les préférences de deux utilisateurs.

In [ ]:
isolated_agent = MemoryAwareSupportAgent()

isolated_agent.receive("alice", "Je préfère les exemples en Python.")
isolated_agent.receive("bob", "Je préfère les exemples en TypeScript.")

alice_response = isolated_agent.receive("alice", "Aide-moi à créer un ticket.")
bob_response = isolated_agent.receive("bob", "Aide-moi à créer un ticket.")

print("Alice:", alice_response)
print("Bob:", bob_response)

## 6. Oubli utilisateur

La mémoire doit pouvoir être supprimée.

`forget_user` supprime :

- mémoire longue ;
- mémoire courte ;
- state courant.

In [ ]:
isolated_agent.forget_user("alice")
print(json.dumps(isolated_agent.build_context("alice"), ensure_ascii=False, indent=2))

## 7. Lancer les tests

Les tests valident la couche applicative sans LLM.

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "test_memory_agent.py"],
    cwd=str(LAB_PATH),
    text=True,
    capture_output=True,
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
assert result.returncode == 0

## 8. Exercices

Travaille ensuite dans `exercises.md`.

Points prioritaires :

1. classer les informations dans la bonne couche mémoire ;
2. concevoir un profil JSON ;
3. définir une politique de promotion ;
4. écrire un test d'isolation utilisateur ;
5. proposer une architecture mémoire pour un assistant de documentation.